In [6]:
from datasets import load_dataset, Dataset
from markdown_it.rules_block import reference

raw_dataset = load_dataset("tner/wnut2017")
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 2395
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1287
    })
})

In [7]:
raw_dataset['train'][0]

{'tokens': ['@paulwalk',
  'It',
  "'s",
  'the',
  'view',
  'from',
  'where',
  'I',
  "'m",
  'living',
  'for',
  'two',
  'weeks',
  '.',
  'Empire',
  'State',
  'Building',
  '=',
  'ESB',
  '.',
  'Pretty',
  'bad',
  'storm',
  'here',
  'last',
  'evening',
  '.'],
 'tags': [12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  3,
  9,
  9,
  12,
  3,
  12,
  12,
  12,
  12,
  12,
  12,
  12,
  12]}

In [9]:
df = Dataset.to_pandas(raw_dataset['train'])

In [10]:
df.head()

,tokens,tags
0,"[@paulwalk, It, 's, the, view, from, where, I,...","[12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 1..."
1,"[From, Green, Newsfeed, :, AHFA, extends, dead...","[12, 12, 12, 12, 2, 12, 12, 12, 12, 12, 12, 12..."
2,"[Pxleyes, Top, 50, Photography, Contest, Pictu...","[0, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]"
3,"[today, is, my, last, day, at, the, office, .]","[12, 12, 12, 12, 12, 12, 12, 12, 12]"
4,"[4Dbling, 's, place, til, monday, ,, party, pa...","[4, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12]"


In [15]:
df.isna().sum()

tokens    0
tags      0
dtype: int64

In [20]:
example = df.loc[0, :]

In [21]:
for word, tag in zip(example['tokens'], example['tags']):
    print(f"{word:20} {tag}")

@paulwalk            12
It                   12
's                   12
the                  12
view                 12
from                 12
where                12
I                    12
'm                   12
living               12
for                  12
two                  12
weeks                12
.                    12
Empire               3
State                9
Building             9
=                    12
ESB                  3
.                    12
Pretty               12
bad                  12
storm                12
here                 12
last                 12
evening              12
.                    12


In [22]:
from datasets import load_dataset_builder

builder = load_dataset_builder("tner/wnut2017")
builder

In [27]:
print(builder.info.features)

{'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'tags': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None)}


In [29]:
raw_dataset["train"].column_names  # ["tokens", "tags"]

['tokens', 'tags']

In [30]:
tag_feature = raw_dataset["train"].features["tags"].feature
print(tag_feature)

Value(dtype='int32', id=None)


In [31]:
word_tags = df[["tokens", "tags"]].explode(["tokens", "tags"])
print(word_tags.head(30))

      tokens tags
0  @paulwalk   12
0         It   12
0         's   12
0        the   12
0       view   12
0       from   12
0      where   12
0          I   12
0         'm   12
0     living   12
0        for   12
0        two   12
0      weeks   12
0          .   12
0     Empire    3
0      State    9
0   Building    9
0          =   12
0        ESB    3
0          .   12
0     Pretty   12
0        bad   12
0      storm   12
0       here   12
0       last   12
0    evening   12
0          .   12
1       From   12
1      Green   12
1   Newsfeed   12


In [32]:
lengths_match = df.apply(
    lambda row: len(row["tokens"]) == len(row["tags"]),
    axis=1,
)
print("Mismatched examples:", (~lengths_match).sum())

Mismatched examples: 0


In [33]:
print("Empty sentences:", df["tokens"].map(len).eq(0).sum())

Empty sentences: 0


In [34]:
from transformers import AutoTokenizer

checkpoint = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [45]:
def tokenize(example):
    encoded = tokenizer(
        example['tokens'],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, tags in enumerate(example['tags']):
        labels = []
        previous_word_id = None

        for word_id in encoded.word_ids(batch_index=i):
            if word_id is None:
                labels.append(-100)
            elif word_id != previous_word_id:
                labels.append(tags[word_id])
            else:
                labels.append(-100)

            previous_word_id = word_id

        all_labels.append(labels)

    encoded['labels'] = all_labels
    return encoded

In [48]:
tokenized_dataset = raw_dataset.map(tokenize, batched=True, remove_columns=raw_dataset['train'].column_names)

Map: 100%|██████████| 1287/1287 [00:00<00:00, 21693.37 examples/s]


In [51]:
num_labels = df["tags"].explode().nunique()
print(num_labels)

13


In [52]:
import evaluate

metric = evaluate.load('seqeval')

In [53]:
label_names = [
    "B-corporation",    # 0
    "B-creative-work",  # 1
    "B-group",          # 2
    "B-location",       # 3
    "B-person",         # 4
    "B-product",        # 5
    "I-corporation",    # 6
    "I-creative-work",  # 7
    "I-group",          # 8
    "I-location",       # 9
    "I-person",         # 10
    "I-product",        # 11
    "O",               # 12
]

In [54]:
import numpy as np

def compute_metric(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }


In [55]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

In [84]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(checkpoint, label2id=label2id, id2label=id2label, attention_probs_dropout_prob=0.2, hidden_dropout_prob=0.2)
data_collator = DataCollatorForTokenClassification(tokenizer)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 676.14it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not o

In [85]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

In [86]:
train_args = TrainingArguments(
    'ner-classifier-practice',
    learning_rate=3e-5,
    weight_decay=0.05,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=0,
    load_best_model_at_end=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    use_cpu=False,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_strategy="steps",
logging_steps=10,
logging_first_step=True,

)

In [87]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metric,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    processing_class=tokenizer,
    data_collator=data_collator
)

In [88]:
print("Device:", trainer.args.device)
print("Classes:", model.config.num_labels)


Device: cuda:0
Classes: 13


In [89]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.157521,0.244364,0.720721,0.382775,0.500000,0.942795
2,0.122801,0.216122,0.498113,0.473684,0.485592,0.947372
3,0.087114,0.259143,0.547965,0.450957,0.494751,0.947944
4,0.042021,0.259651,0.638400,0.477273,0.546201,0.953029
5,0.035319,0.260536,0.639535,0.526316,0.577428,0.954681
6,0.022973,0.265958,0.613699,0.535885,0.572158,0.954872
7,0.014054,0.284891,0.644245,0.515550,0.572757,0.954618
8,0.011510,0.284782,0.589641,0.531100,0.558842,0.954300


C:\Users\user\Desktop\MlLibraries\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', '

TrainOutput(global_step=1200, training_loss=0.07469538438444336, metrics={'train_runtime': 369.2089, 'train_samples_per_second': 51.895, 'train_steps_per_second': 3.25, 'total_flos': 606251409795492.0, 'train_loss': 0.07469538438444336, 'epoch': 8.0})

In [78]:
output = trainer.predict(tokenized_dataset["validation"])

predictions = output.predictions.argmax(axis=-1)
labels = output.label_ids

In [79]:
from seqeval.metrics import classification_report

true_labels = [
    [label_names[l] for l in sentence if l != -100]
    for sentence in labels
]

true_predictions = [
    [label_names[p] for p, l in zip(pred, gold) if l != -100]
    for pred, gold in zip(predictions, labels)
]

print(classification_report(
    true_labels,
    true_predictions,
    digits=3,
    zero_division=0,
))

               precision    recall  f1-score   support

  corporation      0.281     0.471     0.352        34
creative-work      0.500     0.171     0.255       105
        group      0.190     0.103     0.133        39
     location      0.571     0.541     0.556        74
       person      0.758     0.713     0.735       470
      product      0.294     0.263     0.278       114

    micro avg      0.609     0.530     0.566       836
    macro avg      0.432     0.377     0.385       836
 weighted avg      0.600     0.530     0.553       836



In [80]:
from collections import Counter

train_entities = Counter(
    label_names[tag][2:]
    for tags in raw_dataset["train"]["tags"]
    for tag in tags
    if label_names[tag].startswith("B-")
)

print(train_entities)

Counter({'person': 660, 'location': 548, 'group': 264, 'corporation': 221, 'product': 142, 'creative-work': 140})


In [81]:
shown = 0

for i, (gold, pred) in enumerate(zip(true_labels, true_predictions)):
    has_group_error = any(
        g != p and (g.endswith("-group") or p.endswith("-group"))
        for g, p in zip(gold, pred)
    )

    if not has_group_error:
        continue

    words = raw_dataset["validation"][i]["tokens"]
    print("\nSentence:", " ".join(words))

    for word, g, p in zip(words, gold, pred):
        marker = " <-- ERROR" if g != p else ""
        print(f"{word:20} true={g:18} pred={p:18}{marker}")

    shown += 1
    if shown == 3:
        break


Sentence: Warriors looked a little bored , gave a c plus effort and got the win . . . This should be short and sweet .
Warriors             true=B-group            pred=O                  <-- ERROR
looked               true=O                  pred=O                 
a                    true=O                  pred=O                 
little               true=O                  pred=O                 
bored                true=O                  pred=O                 
,                    true=O                  pred=O                 
gave                 true=O                  pred=O                 
a                    true=O                  pred=O                 
c                    true=O                  pred=O                 
plus                 true=O                  pred=O                 
effort               true=O                  pred=O                 
and                  true=O                  pred=O                 
got                  true=O               

In [82]:
from collections import Counter

group_predictions = Counter(
    p
    for gold, pred in zip(true_labels, true_predictions)
    for g, p in zip(gold, pred)
    if g in {"B-group", "I-group"}
)

print(group_predictions)

Counter({'O': 41, 'B-group': 6, 'B-corporation': 5, 'B-person': 4, 'I-group': 3, 'B-product': 1, 'I-product': 1, 'B-creative-work': 1, 'I-creative-work': 1, 'B-location': 1})


In [90]:
print("Selection metric:", trainer.args.metric_for_best_model)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best score:", trainer.state.best_metric)

best_model = trainer.model

Selection metric: f1
Best checkpoint: ner-classifier-practice\checkpoint-750
Best score: 0.5774278215223096


In [91]:
validation_metrics = trainer.evaluate(
    eval_dataset=tokenized_dataset["validation"],
)

print(validation_metrics)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.011510,0.260650,8,0.637681,0.526316,0.576671,0.954618


{'eval_loss': 0.2606503665447235, 'eval_precision': 0.6376811594202898, 'eval_recall': 0.5263157894736842, 'eval_f1': 0.5766710353866317, 'eval_accuracy': 0.9546176825780207}


In [92]:
output = trainer.predict(tokenized_dataset["validation"])

predictions = output.predictions.argmax(axis=-1)
labels = output.label_ids

In [93]:
trainer.save_model("./best-ner-model")
tokenizer.save_pretrained("./best-ner-model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


('./best-ner-model\\tokenizer_config.json', './best-ner-model\\tokenizer.json')